In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
init_load_flag=int(dbutils.widgets.get("init_load_flag"))

### **Data Reading**

In [0]:
df=spark.sql("select * from databricks_cat.silver.customers_silver")
df.display()

### **Removing Duplicates**

In [0]:
df=df.dropDuplicates(subset=['customer_id'])
df.display()

# **Dividing New vs Old Records**

In [0]:
if init_load_flag==0:
    df_old=spark.sql('''select DimCustomerKey,customer_id,create_date,update_date
                 from databricks_cat.gold.DimCustomers''')
else:
    df_old=spark.sql('''select 0 DimCustomerKey,0 customer_id,0 create_date,0 update_date
                 from databricks_cat.silver.customers_silver
                 where 1=0''')

In [0]:
df_old.display()

### **Renaming columns of df_old**

In [0]:
df_old=df_old.withColumnRenamed("DimCustomerKey","old_DimCustomerKey")\
                .withColumnRenamed("customer_id","old_customer_id")\
                .withColumnRenamed("create_date","old_create_date")\
                .withColumnRenamed("update_date","old_update_date")

### **Applying Join with the Old Records**

In [0]:
df_join=df.join(df_old,df.customer_id==df_old.old_customer_id,'left')
df_join.display()

### **Seperating Old vs New Records**

In [0]:
df_new=df_join.filter(df_join.old_DimCustomerKey.isNull())

In [0]:
df_old=df_join.filter(df_join.old_DimCustomerKey.isNotNull())

### **Preparing df_old**

In [0]:
# Removing unwanted columns
df_old=df_old.drop("old_customer_id","old_update_date")
# Updating "old_DimCustomerKey" to "DimCustomerKey"
df_old=df_old.withColumnRenamed("old_DimCustomerKey","DimCustomerKey")
# Renaming "old_create_date" to "create_date" and changing it to timestamp dtype
df_old=df_old.withColumnRenamed("old_create_date","create_date")
df_old=df_old.withColumn("create_date",to_timestamp("create_date"))
# Recreating "update_date" column with current IST timestamp
df_old=df_old.withColumn("update_date",from_utc_timestamp(current_timestamp(), "Asia/Kolkata"))
df_old.display()

### **Preparing df_new**

In [0]:
# Removing unwanted columns
df_new=df_new.drop("old_DimCustomerKey","old_customer_id","old_update_date","old_create_date")
# Recreating "update_date","create_date" column with current_timestamp in IST
df_new=df_new.withColumn("update_date",from_utc_timestamp(current_timestamp(), "Asia/Kolkata"))
df_new=df_new.withColumn("create_date",from_utc_timestamp(current_timestamp(), "Asia/Kolkata"))
df_new.display()

### **Surrogate Key - From 1**

In [0]:
df_new=df_new.withColumn("DimCustomerKey",monotonically_increasing_id()+lit(1))
df_new.display()

### **Adding Max Surrogate Key**

In [0]:
if init_load_flag==1:
    max_surrogate_key=0
else:
    df_maxsur=spark.sql("select max(DimCustomerKey) as max_surrogate_key from databricks_cat.gold.DimCustomers")
    # Converting df_maxsur to max_surrogate_key variable
    max_surrogate_key=df_maxsur.collect()[0]['max_surrogate_key']

In [0]:
df_new=df_new.withColumn("DimCustomerKey",lit(max_surrogate_key)+col("DimCustomerKey"))

### **Union of df_new and df_old**

In [0]:
df_final=df_new.unionByName(df_old)

In [0]:
df_final.display()

### **SCD Type - 1**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricks_cat.gold.DimCustomers"):
    dlt_obj=DeltaTable.forPath(spark,"abfss://gold@databricksetestorage.dfs.core.windows.net/DimCustomers")
    dlt_obj.alias("trg").merge(df_final.alias("src"),"trg.DimCustomerKey==src.DimCustomerKey")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_final.write.mode("overwrite")\
    .format("delta")\
    .option("path","abfss://gold@databricksetestorage.dfs.core.windows.net/DimCustomers")\
    .saveAsTable("databricks_cat.gold.DimCustomers")


In [0]:
%sql
select * from databricks_cat.gold.dimcustomers